In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
import time
import gc
import csv

print("--- STARTING KAGGLE BATCH EXECUTION: ADVANCED CONVLSTM ---")

# ==========================================
# PART 1: CONFIGURATION & DATA PIPELINE
# ==========================================
DATASET_PATH = "/kaggle/input/datasets/yashaswi15/new-training-data-aerosense/Delhi_NCR_Advanced_10Ch_Cube.npy"

BATCH_SIZE = 2  # Keeping at 2 to prevent OOM with 5x5 ConvLSTM kernels

print(f"Loading 10-Channel DataCube from: {DATASET_PATH}")
master_data = np.load(DATASET_PATH, mmap_mode='r')
total_days = master_data.shape[0]
train_split = int(total_days * 0.8) 

class AdvancedDataGenerator(Sequence):
    def __init__(self, data_cube, start_idx, end_idx, batch_size=2):
        self.data = data_cube
        self.start_idx = start_idx
        self.end_idx = end_idx - 8 
        self.batch_size = batch_size
        self.indices = np.arange(self.start_idx, self.end_idx)

    def __len__(self):
        return int(np.floor(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        X, Y = [], []
        
        for i in batch_indices:
            # X: 7 days lookback, all 10 channels
            X.append(self.data[i : i+7]) 
            # Y: 8th day, ONLY Carbon Monoxide (Channel Index 2)
            Y.append(self.data[i+7, :, :, 2:3]) 
            
        # Nan-to-num for absolute safety during training
        X_batch = np.nan_to_num(np.array(X), nan=0.0).astype('float16')
        Y_batch = np.nan_to_num(np.array(Y), nan=0.0).astype('float16')
        return X_batch, Y_batch

print("Initializing Training and Validation Generators...")
train_gen = AdvancedDataGenerator(master_data, 0, train_split, batch_size=BATCH_SIZE)
val_gen = AdvancedDataGenerator(master_data, train_split, total_days, batch_size=BATCH_SIZE)


# ==========================================
# PART 2: ADVANCED CONVLSTM ARCHITECTURE
# ==========================================
print("\n[2/3] Building 5x5 ConvLSTM Architecture...")

def build_advanced_convlstm(input_shape=(7, 141, 231, 10)):
    inputs = layers.Input(shape=input_shape)
    
    # Layer 1: Large Receptive Field for High-Speed Wind Advection
    x = layers.ConvLSTM2D(
        filters=32, kernel_size=(5, 5), padding='same',
        return_sequences=True, activation='tanh', recurrent_dropout=0.0
    )(inputs)
    x = layers.BatchNormalization()(x)
    
    # Layer 2: Standard Kernel for Local Pollution Diffusion
    x = layers.ConvLSTM2D(
        filters=64, kernel_size=(3, 3), padding='same',
        return_sequences=False, activation='tanh'
    )(x)
    x = layers.BatchNormalization()(x)
    
    # Final Output Layer (Predicting 1 Channel: CO)
    outputs = layers.Conv2D(1, (1, 1), activation='linear')(x)
    
    return Model(inputs=inputs, outputs=outputs, name="Advanced_ConvLSTM_10Ch")

model = build_advanced_convlstm()


import csv 

# ==========================================
# PART 3: ADVANCED CUSTOM TRAINING LOOP (WITH NATIVE CSV LOGGER)
# ==========================================
MODEL_NAME = "Advanced_ConvLSTM" 
MODEL_SAVE_PATH = f"/kaggle/working/Delhi_NCR_{MODEL_NAME}_Best.keras"
CSV_LOG_PATH = f"/kaggle/working/{MODEL_NAME}_Training_Log.csv"

# Create the CSV file and write the headers
with open(CSV_LOG_PATH, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss", "val_mape"])

print(f"\n[3/3] Initiating Custom Training Loop for {MODEL_NAME}...")
print(f" -> Metrics will be safely logged to: {CSV_LOG_PATH}")

initial_learning_rate = 0.001
decay_steps = 50 * len(train_gen)
lr_schedule = CosineDecay(initial_learning_rate, decay_steps)
optimizer = Adam(learning_rate=lr_schedule, clipnorm=1.0) # Crucial for LSTMs!

def calculate_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Dynamic threshold for 'high pollution' zones
    threshold = tf.reduce_mean(y_true)
    weights = tf.where(y_true > threshold, 3.0, 1.0)
    
    # Returns the weighted MAE
    return tf.reduce_mean(weights * tf.abs(y_true - y_pred))

@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predictions = model(x_batch, training=True)
        loss = calculate_loss(y_batch, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

@tf.function
def val_step(x_batch, y_batch):
    predictions = model(x_batch, training=False)
    val_loss = calculate_loss(y_batch, predictions)
    
    # Mathematically calculate MAPE to save to the CSV
    y_true_f = tf.cast(y_batch, tf.float32)
    pred_f = tf.cast(predictions, tf.float32)
    mape = tf.reduce_mean(tf.abs((y_true_f - pred_f) / (y_true_f + 1e-10))) * 100.0
    
    return val_loss, mape

EPOCHS = 50
best_val_loss = float('inf')
patience = 7
patience_counter = 0

print(f"\n--- Starting 50-Epoch Backpropagation ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    epoch_loss_avg = tf.keras.metrics.Mean()
    epoch_val_loss_avg = tf.keras.metrics.Mean()
    epoch_val_mape_avg = tf.keras.metrics.Mean() 
    
    for step in range(len(train_gen)):
        x_batch, y_batch = train_gen[step]
        loss_val = train_step(x_batch, y_batch)
        epoch_loss_avg.update_state(loss_val)
        
    # --- FIX APPLIED HERE: Changed test_gen to val_gen ---
    for step in range(len(val_gen)):
        x_val, y_val = val_gen[step]
        v_loss, v_mape = val_step(x_val, y_val)
        epoch_val_loss_avg.update_state(v_loss)
        epoch_val_mape_avg.update_state(v_mape)
        
    train_loss = epoch_loss_avg.result().numpy()
    val_loss = epoch_val_loss_avg.result().numpy()
    val_mape = epoch_val_mape_avg.result().numpy()
    
    current_lr = optimizer.learning_rate(optimizer.iterations).numpy() if callable(optimizer.learning_rate) else optimizer.learning_rate.numpy()
        
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Time: {time.time() - start_time:.1f}s | LR: {current_lr:.5f} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f} | Val MAPE: {val_mape:.2f}%")
    
    # APPEND METRICS TO NATIVE CSV FILE
    with open(CSV_LOG_PATH, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch + 1, train_loss, val_loss, val_mape])
    
    if val_loss < best_val_loss:
        print(f"  -> Val Loss improved from {best_val_loss:.5f} to {val_loss:.5f}. Saving weights!")
        best_val_loss = val_loss
        model.save(MODEL_SAVE_PATH)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n[!] Early Stopping Triggered.")
            break
            
    gc.collect()
    tf.keras.backend.clear_session()

print(f"\nTraining Complete! Logs successfully written to {CSV_LOG_PATH}")

2026-04-23 16:31:00.218570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776961860.389811      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776961860.444214      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776961860.864205      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776961860.864261      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776961860.864264      23 computation_placer.cc:177] computation placer alr

--- STARTING KAGGLE BATCH EXECUTION: ADVANCED CONVLSTM ---
Loading 10-Channel DataCube from: /kaggle/input/datasets/yashaswi15/new-training-data-aerosense/Delhi_NCR_Advanced_10Ch_Cube.npy
Initializing Training and Validation Generators...

[2/3] Building 5x5 ConvLSTM Architecture...


I0000 00:00:1776961886.950573      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



[3/3] Initiating Custom Training Loop for Advanced_ConvLSTM...
 -> Metrics will be safely logged to: /kaggle/working/Advanced_ConvLSTM_Training_Log.csv

--- Starting 50-Epoch Backpropagation ---


E0000 00:00:1776961891.607729      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'Func/gradient_tape/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while_grad/body/_267/input/_766' -> 'gradient_tape/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while_grad/body/_267/gradient_tape/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/gradients/AddN', 'Func/gradient_tape/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1/while/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1/while_grad/body/_417/input/_855' -> 'gradient_tape/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1/while/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1/while_grad/body/_417/gradient_tape/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1/while/gradients/AddN', 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_129/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/Sigmoid' -> 'Advanced_C

...

Epoch 22/50 | Time: 312.7s | LR: 0.00059 | Train Loss: 0.08008 | Val Loss: 0.09308 | Val MAPE: 110395488.00%
  -> No improvement. Patience: 5/7


E0000 00:00:1776969118.378496      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/convolution_7' -> 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/ArithmeticOptimizer/AddOpsRewrite_Leaf_1_add_8', 'Func/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/input/_140' -> 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/mul', 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/Tanh_1' -> 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/mul_2', 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1/while/body/_1

Epoch 23/50 | Time: 312.5s | LR: 0.00056 | Train Loss: 0.08035 | Val Loss: 0.08321 | Val MAPE: 87784008.00%
  -> No improvement. Patience: 6/7


E0000 00:00:1776969431.811678      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/Tanh_1' -> 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/mul_2', 'Func/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/input/_140' -> 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/mul', 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/convolution_7' -> 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/body/_44/Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1_2/while/conv_lstm_cell_1/ArithmeticOptimizer/AddOpsRewrite_Leaf_1_add_8', 'Advanced_ConvLSTM_10Ch_1/conv_lstm2d_1/while/body/_1

Epoch 24/50 | Time: 312.9s | LR: 0.00053 | Train Loss: 0.08022 | Val Loss: 0.08611 | Val MAPE: 191550640.00%
  -> No improvement. Patience: 7/7

[!] Early Stopping Triggered.

Training Complete! Logs successfully written to /kaggle/working/Advanced_ConvLSTM_Training_Log.csv
